# Q13: population dimensionality from stimulus presentation to outcome

This analysis uses exactly the filtered Q11/Q12 trial table. For every mouse
and brain area, one trial is represented by the vector of mean firing rates
in the half-open interval `[stim_start, outcome)`. Neurons are standardized
across trials before PCA so that dimensionality reflects population
covariation rather than absolute firing-rate scale.

The primary dimensionality measure is the PCA participation ratio. Neuron
count is matched across the three brain areas separately within each mouse,
using repeated random subsampling. The number of components explaining at
least 90% of variance and participation ratio divided by sampled population
size are saved as sensitivity measures.

In [ ]:
import csv
import json
import os
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

In [ ]:
project_folder = Path(os.environ.get("CON2PHYS_PROJECT_DIR", "E:/con2phys_submission"))
repository_folder = Path(os.environ.get(
    "CON2PHYS_REPOSITORY_DIR", "D:/git_repos/con2phys_submission"
))
sua_csv_path = project_folder / "dataset" / "full_sua_dataset.csv"
trials_csv_path = repository_folder / "results" / "q11" / "trials_info_filtered.csv"
q_path = Path(os.environ.get("Q13_OUTPUT_DIR", repository_folder / "results" / "q13"))

N_NEURON_SUBSAMPLES = int(os.environ.get("Q13_N_SUBSAMPLES", 100))
RANDOM_SEED = 42
EXPECTED_N_MICE = 18
EXPECTED_N_UNITS = 1_449
EXPECTED_BRAIN_AREAS = (1, 2, 3)
MAX_MICE = int(os.environ["Q13_MAX_MICE"]) if os.environ.get("Q13_MAX_MICE") else None

run_tag = "full" if MAX_MICE is None else f"first_{MAX_MICE}_mice"
file_suffix = "" if MAX_MICE is None else f"_{run_tag}"
unit_counts_csv_path = q_path / f"q13_unit_counts{file_suffix}.csv"
subsamples_csv_path = q_path / f"q13_dimensionality_subsamples{file_suffix}.csv"
mouse_area_csv_path = q_path / f"q13_dimensionality_mouse_area{file_suffix}.csv"
area_summary_csv_path = q_path / f"q13_dimensionality_area_summary{file_suffix}.csv"
figure_path = q_path / f"q13_dimensionality_by_area{file_suffix}.png"
config_path = q_path / f"q13_run_config{file_suffix}.json"

RUN_CONFIG = {
    "feature": "trial_level_mean_firing_rate_stim_start_to_outcome",
    "spike_window": "[stim_start, outcome)",
    "trial_filtering": "reuse results/q11/trials_info_filtered.csv from Q11/Q12",
    "neuron_standardization": "z_score_across_trials_ddof_1; zero_variance_neurons_set_to_zero",
    "primary_metric": "pca_participation_ratio",
    "sensitivity_metrics": ["participation_ratio_per_sampled_neuron", "n_components_90pct"],
    "neuron_matching": "minimum_area_population_within_mouse",
    "n_neuron_subsamples": N_NEURON_SUBSAMPLES,
    "random_seed": RANDOM_SEED,
    "max_mice": MAX_MICE,
}
RUN_CONFIG

## Validate the filtered trial table

No new behavioral trial filtering is performed here. The notebook validates
and directly reuses the Q11/Q12 filtered trial identifiers.

In [ ]:
required_paths = [sua_csv_path, trials_csv_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required paths not found: {missing_paths}")

trials_df = pd.read_csv(trials_csv_path)
expected_trial_columns = [
    "mouse_id", "trial_id", "variable_A", "variable_B", "variable_C",
    "trial_start", "stim_start", "outcome", "trial_end", "trial_duration",
]
if list(trials_df.columns) != expected_trial_columns:
    raise ValueError(f"Unexpected trial schema: {list(trials_df.columns)}")
if trials_df[expected_trial_columns].isna().any().any():
    raise ValueError("Filtered trial table contains missing values")
if trials_df[["mouse_id", "trial_id"]].duplicated().any():
    raise ValueError("Duplicate mouse/trial identifiers found")

correct_order = (
    trials_df["trial_start"].lt(trials_df["stim_start"])
    & trials_df["stim_start"].lt(trials_df["outcome"])
    & trials_df["outcome"].lt(trials_df["trial_end"])
)
if not correct_order.all():
    raise ValueError("Filtered trial table contains incorrectly ordered events")

all_mouse_ids = sorted(int(value) for value in trials_df["mouse_id"].unique())
if len(all_mouse_ids) != EXPECTED_N_MICE:
    raise ValueError(f"Expected {EXPECTED_N_MICE} mice, found {len(all_mouse_ids)}")
selected_mouse_ids = all_mouse_ids if MAX_MICE is None else all_mouse_ids[:MAX_MICE]
trials_df = (
    trials_df.loc[trials_df["mouse_id"].isin(selected_mouse_ids)]
    .sort_values(["mouse_id", "trial_id"])
    .reset_index(drop=True)
)
trials_by_mouse = {
    int(mouse_id): group.reset_index(drop=True)
    for mouse_id, group in trials_df.groupby("mouse_id", sort=True)
}
trial_counts = trials_df.groupby("mouse_id").size().rename("n_trials")
display(trial_counts.to_frame())
print(f"Validated and reused {len(trials_df):,} filtered trials from {len(selected_mouse_ids)} mice.")

## Construct trial-by-neuron mean-rate matrices

In [ ]:
expected_sua_columns = [
    "mouse_id", "cluster_id", "brain_area", "recording_duration_sec",
    "unit_last_spike_sec", "n_spikes", "spikes",
]
rate_columns_by_group = defaultdict(list)
cluster_ids_by_group = defaultdict(list)
unit_keys = set()
n_source_units = 0

csv.field_size_limit(2**31 - 1)
load_started = time.perf_counter()
with sua_csv_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if reader.fieldnames != expected_sua_columns:
        raise ValueError(f"Unexpected SUA schema: {reader.fieldnames}")

    for row in reader:
        n_source_units += 1
        mouse_id = int(row["mouse_id"])
        if mouse_id not in trials_by_mouse:
            continue

        cluster_id = int(row["cluster_id"])
        brain_area = int(row["brain_area"])
        unit_key = (mouse_id, cluster_id)
        group_key = (mouse_id, brain_area)
        if unit_key in unit_keys:
            raise ValueError(f"Duplicate unit key: {unit_key}")
        if brain_area not in EXPECTED_BRAIN_AREAS:
            raise ValueError(f"Unexpected brain area for {unit_key}: {brain_area}")

        spike_times = np.fromstring(row["spikes"], sep=";", dtype=np.float64)
        expected_n_spikes = int(row["n_spikes"])
        if spike_times.size != expected_n_spikes or spike_times.size == 0:
            raise ValueError(f"Spike-count mismatch for {unit_key}")
        if not np.isfinite(spike_times).all() or np.any(np.diff(spike_times) < 0):
            raise ValueError(f"Invalid or unsorted spikes for {unit_key}")

        mouse_trials = trials_by_mouse[mouse_id]
        window_starts = mouse_trials["stim_start"].to_numpy(dtype=float)
        window_ends = mouse_trials["outcome"].to_numpy(dtype=float)
        window_durations = window_ends - window_starts
        if np.any(~np.isfinite(window_durations)) or np.any(window_durations <= 0):
            raise ValueError(f"Invalid analysis-window duration for mouse {mouse_id}")

        left = np.searchsorted(spike_times, window_starts, side="left")
        right = np.searchsorted(spike_times, window_ends, side="left")
        firing_rates = (right - left) / window_durations
        rate_columns_by_group[group_key].append(firing_rates)
        cluster_ids_by_group[group_key].append(cluster_id)
        unit_keys.add(unit_key)

if MAX_MICE is None and n_source_units != EXPECTED_N_UNITS:
    raise ValueError(f"Expected {EXPECTED_N_UNITS} SUA units, found {n_source_units}")

expected_group_keys = {
    (mouse_id, brain_area)
    for mouse_id in selected_mouse_ids
    for brain_area in EXPECTED_BRAIN_AREAS
}
if set(rate_columns_by_group) != expected_group_keys:
    missing_groups = expected_group_keys - set(rate_columns_by_group)
    raise ValueError(f"Missing mouse/area groups: {sorted(missing_groups)}")

firing_rates_by_group = {}
unit_count_rows = []
for group_key in sorted(expected_group_keys):
    mouse_id, brain_area = group_key
    matrix = np.column_stack(rate_columns_by_group[group_key])
    if matrix.shape[0] != len(trials_by_mouse[mouse_id]) or not np.isfinite(matrix).all():
        raise ValueError(f"Invalid firing-rate matrix for {group_key}")
    firing_rates_by_group[group_key] = matrix
    unit_count_rows.append({
        "mouse_id": mouse_id,
        "brain_area": brain_area,
        "n_units": matrix.shape[1],
        "n_trials": matrix.shape[0],
    })

unit_counts_df = pd.DataFrame(unit_count_rows)
matched_units_by_mouse = unit_counts_df.groupby("mouse_id")["n_units"].min().astype(int)
unit_counts_df["n_matched_units"] = unit_counts_df["mouse_id"].map(matched_units_by_mouse)
q_path.mkdir(parents=True, exist_ok=True)
unit_counts_df.to_csv(unit_counts_csv_path, index=False)
print(f"Built {len(firing_rates_by_group)} matrices in {time.perf_counter() - load_started:.1f} s.")
display(unit_counts_df)

## PCA dimensionality with matched neuron subsampling

In [ ]:
def calculate_dimensionality(rate_matrix):
    """Return correlation-PCA dimensionality metrics for trials x neurons."""
    centered = rate_matrix - rate_matrix.mean(axis=0, keepdims=True)
    scales = centered.std(axis=0, ddof=1)
    variable_mask = np.isfinite(scales) & (scales > 0)
    standardized = np.zeros_like(centered, dtype=np.float64)
    standardized[:, variable_mask] = centered[:, variable_mask] / scales[variable_mask]

    covariance = standardized.T @ standardized / (standardized.shape[0] - 1)
    eigenvalues = np.linalg.eigvalsh(covariance)[::-1]
    eigenvalues = np.clip(eigenvalues, 0.0, None)
    total_variance = eigenvalues.sum()
    squared_sum = np.square(eigenvalues).sum()
    if total_variance <= 0 or squared_sum <= 0:
        raise ValueError("Population has no variance across trials")

    participation_ratio = total_variance**2 / squared_sum
    cumulative_variance = np.cumsum(eigenvalues) / total_variance
    n_components_90 = int(np.searchsorted(cumulative_variance, 0.90, side="left") + 1)
    return {
        "participation_ratio": float(participation_ratio),
        "normalized_participation_ratio": float(participation_ratio / rate_matrix.shape[1]),
        "n_components_90": n_components_90,
        "n_variable_units": int(variable_mask.sum()),
    }


rng = np.random.default_rng(RANDOM_SEED)
subsample_rows = []
analysis_started = time.perf_counter()
for mouse_id in selected_mouse_ids:
    n_matched_units = int(matched_units_by_mouse.loc[mouse_id])
    for brain_area in EXPECTED_BRAIN_AREAS:
        matrix = firing_rates_by_group[(mouse_id, brain_area)]
        n_available_units = matrix.shape[1]
        for subsample_id in range(1, N_NEURON_SUBSAMPLES + 1):
            if n_available_units == n_matched_units:
                selected_columns = np.arange(n_available_units)
            else:
                selected_columns = rng.choice(
                    n_available_units, size=n_matched_units, replace=False
                )
            metrics = calculate_dimensionality(matrix[:, selected_columns])
            subsample_rows.append({
                "mouse_id": mouse_id,
                "brain_area": brain_area,
                "subsample_id": subsample_id,
                "n_trials": matrix.shape[0],
                "n_available_units": n_available_units,
                "n_sampled_units": n_matched_units,
                **metrics,
            })

subsamples_df = pd.DataFrame(subsample_rows)
expected_rows = len(selected_mouse_ids) * len(EXPECTED_BRAIN_AREAS) * N_NEURON_SUBSAMPLES
if len(subsamples_df) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} subsample rows, found {len(subsamples_df)}")
if not np.isfinite(subsamples_df.select_dtypes(include=[np.number])).all().all():
    raise ValueError("Non-finite dimensionality result found")
subsamples_df.to_csv(subsamples_csv_path, index=False)
print(f"Calculated {len(subsamples_df):,} PCA subsamples in {time.perf_counter() - analysis_started:.1f} s.")
display(subsamples_df.head())

## Mouse-level estimates, summaries, and diagnostic plot

In [ ]:
mouse_area_df = (
    subsamples_df.groupby(["mouse_id", "brain_area"], as_index=False)
    .agg(
        mean_participation_ratio=("participation_ratio", "mean"),
        sd_participation_ratio=("participation_ratio", "std"),
        mean_normalized_participation_ratio=("normalized_participation_ratio", "mean"),
        sd_normalized_participation_ratio=("normalized_participation_ratio", "std"),
        mean_n_components_90=("n_components_90", "mean"),
        sd_n_components_90=("n_components_90", "std"),
        mean_n_variable_units=("n_variable_units", "mean"),
        n_trials=("n_trials", "first"),
        n_available_units=("n_available_units", "first"),
        n_sampled_units=("n_sampled_units", "first"),
        n_subsamples=("subsample_id", "size"),
    )
)
if len(mouse_area_df) != len(selected_mouse_ids) * len(EXPECTED_BRAIN_AREAS):
    raise RuntimeError("Mouse-area aggregation has an unexpected number of rows")
if not (mouse_area_df.groupby("mouse_id")["n_sampled_units"].nunique() == 1).all():
    raise RuntimeError("Neuron count was not matched across areas within mouse")
mouse_area_df.to_csv(mouse_area_csv_path, index=False)

area_summary_df = (
    mouse_area_df.groupby("brain_area", as_index=False)
    .agg(
        mean_participation_ratio=("mean_participation_ratio", "mean"),
        sd_participation_ratio=("mean_participation_ratio", "std"),
        mean_normalized_participation_ratio=("mean_normalized_participation_ratio", "mean"),
        sd_normalized_participation_ratio=("mean_normalized_participation_ratio", "std"),
        mean_n_components_90=("mean_n_components_90", "mean"),
        sd_n_components_90=("mean_n_components_90", "std"),
        n_mice=("mouse_id", "nunique"),
    )
)
area_summary_df["sem_participation_ratio"] = (
    area_summary_df["sd_participation_ratio"] / np.sqrt(area_summary_df["n_mice"])
)
area_summary_df["ci95_low"] = (
    area_summary_df["mean_participation_ratio"] - 1.96 * area_summary_df["sem_participation_ratio"]
)
area_summary_df["ci95_high"] = (
    area_summary_df["mean_participation_ratio"] + 1.96 * area_summary_df["sem_participation_ratio"]
)
area_summary_df.to_csv(area_summary_csv_path, index=False)
with config_path.open("w", encoding="utf-8") as handle:
    json.dump(RUN_CONFIG, handle, indent=2)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for _, mouse_df in mouse_area_df.groupby("mouse_id", sort=True):
    mouse_df = mouse_df.sort_values("brain_area")
    ax.plot(
        mouse_df["brain_area"], mouse_df["mean_participation_ratio"],
        color="0.75", marker="o", markersize=3, linewidth=0.8, alpha=0.75,
    )
ax.errorbar(
    area_summary_df["brain_area"], area_summary_df["mean_participation_ratio"],
    yerr=1.96 * area_summary_df["sem_participation_ratio"],
    fmt="o-", color="black", markersize=7, capsize=4, linewidth=1.7,
    label="Mean ± 95% CI",
)
ax.set_xticks(EXPECTED_BRAIN_AREAS, [f"Brain area {area}" for area in EXPECTED_BRAIN_AREAS])
ax.set_xlabel("Brain area")
ax.set_ylabel("PCA participation ratio")
ax.set_title("Stimulus-to-outcome population dimensionality")
ax.legend(frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

display(mouse_area_df)
display(area_summary_df)
print(f"Saved Q13 notebook results to {q_path}")